In [0]:
# S'assurer que le schéma silver existe dans le catalogue
spark.sql("USE CATALOG dbw_lab;")
spark.sql("CREATE SCHEMA IF NOT EXISTS dbw_lab.silver;")

In [0]:
from pyspark.sql.functions import col, coalesce
import pyspark.sql.types

In [0]:
# Liste exacte de tes tables Bronze (avec le préfixe)
bronze_tables = [
    "olist_customers",
    "olist_geolocation",
    "olist_order_items",
    "olist_order_payments",
    "olist_order_reviews",
    "olist_orders",
    "olist_products",
    "olist_sellers",
    "product_category_name_translation"
]

for source_table in bronze_tables:
    # Nettoyage du nom pour la couche Silver (enlève "olist_")
    target_name = source_table.replace("olist_", "")
    
    print(f"Traitement de dbw_lab.bronze.{source_table} -> vers dbw_lab.silver.{target_name}...")
    
    # 1. Lecture depuis la couche Bronze
    df = spark.table(f"dbw_lab.bronze.{source_table}")
    
    # 2. Suppression des doublons stricts
    df_silver = df.dropDuplicates()
    
    # 3. Remplissage intelligent basé sur le nouveau nom cible
    if target_name == "orders":
        df_silver = df_silver.withColumn(
            "order_approved_at",
            coalesce(col("order_approved_at"), col("order_purchase_timestamp"))
        )
        
    elif target_name == "products":
        df_silver = df_silver.fillna({
            "product_weight_g": 1500,
            "product_length_cm": 30,
            "product_height_cm": 15,
            "product_width_cm": 20
        })
    
    # 4. Remplissage générique dynamique
    numeric_cols = [f.name for f in df_silver.schema.fields if isinstance(f.dataType, (pyspark.sql.types.DoubleType, pyspark.sql.types.IntegerType, pyspark.sql.types.LongType, pyspark.sql.types.FloatType))]
    string_cols = [f.name for f in df_silver.schema.fields if isinstance(f.dataType, pyspark.sql.types.StringType)]
    
    if numeric_cols:
        df_silver = df_silver.fillna(0, subset=numeric_cols)
    if string_cols:
        df_silver = df_silver.fillna("inconnu", subset=string_cols)
        
    # 5. Écriture dans la couche Silver avec le nom épuré
    target_table_path = f"dbw_lab.silver.{target_name}"
    df_silver.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(target_table_path)
        
    print(f"✅ Table {target_table_path} sauvegardée avec succès !")

print("🚀 Transformation terminée ! Tes tables Silver portent désormais les bons noms.")

In [0]:
%pip install azure-storage-blob python-dotenv

from dotenv import load_dotenv
import os
from azure.storage.blob import BlobServiceClient

# 1. Charger les variables du fichier .env
load_dotenv()
storage_account_name = os.getenv("AZURE_STORAGE_ACCOUNT")
sas_token = os.getenv("AZURE_SAS_TOKEN")
container_name = "silver" # On cible explicitement le conteneur silver

# 2. URL de connexion au service Blob avec le jeton SAS
account_url = f"https://{storage_account_name}.blob.core.windows.net?{sas_token}"
blob_service_client = BlobServiceClient(account_url=account_url)
container_client = blob_service_client.get_container_client(container_name)

# 3. Création d'un dossier temporaire sur le driver pour stocker les fichiers avant l'envoi
temp_local_dir = "/tmp/silver_exports"
os.makedirs(temp_local_dir, exist_ok=True)

# 4. Liste des tables Silver nettoyées (sans le préfixe olist_ comme tu l'as demandé)
tables_silver = [
    "customers",
    "geolocation",
    "order_items",
    "order_payments",
    "order_reviews",
    "orders",
    "products",
    "sellers",
    "product_category_name_translation"
]

for table_name in tables_silver:
    print(f"Préparation et téléversement de la table {table_name}...")
    
    # Étape A : Lire la table depuis Unity Catalog
    df_spark = spark.table(f"dbw_lab.silver.{table_name}")
    
    # Étape B : Convertir en Pandas et sauvegarder en fichier local CSV unique
    local_file_path = os.path.join(temp_local_dir, f"{table_name}.csv")
    df_spark.toPandas().to_csv(local_file_path, index=False)
    
    # Étape C : Téléverser le fichier généré vers Azure Blob Storage
    blob_name = f"cleaned_csv/{table_name}.csv" # On les place dans un dossier cleaned_csv
    blob_client = container_client.get_blob_client(blob_name)
    
    with open(local_file_path, "rb") as data:
        blob_client.upload_blob(data, overwrite=True)
        
    print(f"✅ {table_name} téléversé avec succès dans le conteneur {container_name} !")

print("🚀 Toutes les tables de la couche Silver ont été sauvegardées avec succès dans le cloud Azure !")